# AI Infrastructure Market Pulse

This notebook uses real public market-price data as a proxy for market attention around AI infrastructure. It does not claim to measure audited revenue, capex, or valuation.

No synthetic fallback is used. If market data cannot be fetched, the notebook stops.


In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    relative_path = path.relative_to(repo_root).as_posix()
    print(f"saved: {relative_path}")


## 1. Fetch real prices through yfinance


In [2]:
try:
    import yfinance as yf
except Exception as exc:
    raise ImportError("Install yfinance to run this notebook: python -m pip install yfinance") from exc

tickers = ["NVDA", "AMD", "AVGO", "MSFT", "GOOGL", "AMZN", "META", "TSLA"]
raw = yf.download(tickers, start="2024-01-01", progress=False, auto_adjust=True)["Close"]
if raw.empty:
    raise HotTrendDataError("yfinance returned no real market data")
prices = raw.reset_index().melt(id_vars="Date", var_name="ticker", value_name="price").rename(columns={"Date": "date"})
prices = prices.dropna(subset=["price"])
prices.head(20)


,date,ticker,price
0,2024-01-02,AMD,138.580002
1,2024-01-03,AMD,135.320007
2,2024-01-04,AMD,136.009995
3,2024-01-05,AMD,138.580002
4,2024-01-08,AMD,146.179993
5,2024-01-09,AMD,149.259995
6,2024-01-10,AMD,148.539993
7,2024-01-11,AMD,148.020004
8,2024-01-12,AMD,146.559998
9,2024-01-16,AMD,158.740005


## 2. Audit real price table


In [3]:
audit = source_audit_table(prices, value_col="price", entity_col="ticker", time_col="date")
audit


,series,first_timestamp,last_timestamp,observations,missing_ratio,min_value,max_value
0,AMD,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,78.209999,470.299988
1,AMZN,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,144.570007,274.989990
2,AVGO,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,102.365158,439.790009
3,GOOGL,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,130.322891,402.619995
4,META,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,341.787872,788.148987
5,MSFT,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,351.105774,538.658569
6,NVDA,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,47.539936,235.740005
7,TSLA,2024-01-02 00:00:00,2026-05-22 00:00:00,600,0.0,142.050003,489.880005


## 3. Decompose normalized log prices


In [4]:
components = decompose_table(prices, entity_col="ticker", time_col="date", value_col="price", method="MA_BASELINE", period=21, trend_window=63, transform="log")
summary = editorial_priority(component_summary(components, entity_col="ticker", time_col="date"), entity_col="ticker")
summary


,ticker,observations,first_timestamp,last_timestamp,trend_last,trend_slope_per_step,cycle_strength_proxy,residual_std,max_abs_residual_z,method,trend_slope_per_step_rank_pct,cycle_strength_proxy_rank_pct,max_abs_residual_z_rank_pct,editorial_priority_score
2,AVGO,600,2024-01-02 00:00:00,2026-05-22 00:00:00,3.055940,0.002033,-0.515521,0.492221,31.531357,MA_BASELINE,1.000,1.000,0.500,0.77500
3,GOOGL,600,2024-01-02 00:00:00,2026-05-22 00:00:00,2.996481,0.001351,-1.894500,0.500785,39.031779,MA_BASELINE,0.625,0.500,1.000,0.76875
6,NVDA,600,2024-01-02 00:00:00,2026-05-22 00:00:00,2.711011,0.001477,-0.731154,0.425073,30.558266,MA_BASELINE,0.750,0.875,0.375,0.60625
7,TSLA,600,2024-01-02 00:00:00,2026-05-22 00:00:00,3.038077,0.001532,-1.560344,0.528286,30.540196,MA_BASELINE,0.875,0.750,0.250,0.56875
4,META,600,2024-01-02 00:00:00,2026-05-22 00:00:00,3.277518,0.000695,-8.994485,0.545147,34.014586,MA_BASELINE,0.500,0.375,0.625,0.53125
1,AMZN,600,2024-01-02 00:00:00,2026-05-22 00:00:00,2.824453,0.000536,-12.696403,0.480041,38.471005,MA_BASELINE,0.250,0.250,0.750,0.47500
5,MSFT,600,2024-01-02 00:00:00,2026-05-22 00:00:00,3.058538,0.000199,-28.802148,0.536890,38.798773,MA_BASELINE,0.125,0.125,0.875,0.46250
0,AMD,600,2024-01-02 00:00:00,2026-05-22 00:00:00,2.974027,0.000667,-1.836608,0.521787,29.811574,MA_BASELINE,0.375,0.625,0.125,0.31250


## 4. Cross-sectional AI infrastructure table


In [5]:
latest = prices.sort_values("date").groupby("ticker").tail(1).rename(columns={"price": "latest_price"})
first = prices.sort_values("date").groupby("ticker").head(1).rename(columns={"price": "first_price"})[["ticker", "first_price"]]
returns = latest.merge(first, on="ticker", how="left")
returns["total_return_proxy"] = returns["latest_price"] / returns["first_price"] - 1.0
returns.sort_values("total_return_proxy", ascending=False)


,date,ticker,latest_price,first_price,total_return_proxy
6,2026-05-22,NVDA,216.990005,48.138573,3.507612
3,2026-05-22,AVGO,412.519989,105.914230,2.894850
5,2026-05-22,AMD,470.299988,138.580002,2.393707
0,2026-05-22,GOOGL,386.700012,137.037384,1.821858
4,2026-05-22,AMZN,269.119995,149.929993,0.794971
2,2026-05-22,META,609.140076,343.593658,0.772850
7,2026-05-22,TSLA,430.350006,248.419998,0.732348
1,2026-05-22,MSFT,420.320099,363.801483,0.155356


## 5. Residual events


In [6]:
events = residual_event_table(components, entity_col="ticker", time_col="date", top_n=25)
events


,date,ticker,observed,trend,season,residual,residual_z,abs_residual_z,method
0,2026-05-22,GOOGL,5.957649,2.996481,0.040953,2.920216,39.031779,39.031779,MA_BASELINE
1,2026-05-22,MSFT,6.041017,3.058538,0.042084,2.940395,38.798773,38.798773,MA_BASELINE
2,2026-05-22,AMZN,5.595157,2.824453,0.033841,2.736863,38.471005,38.471005,MA_BASELINE
3,2026-05-21,GOOGL,5.960129,3.087908,0.038738,2.833483,37.868763,37.868763,MA_BASELINE
4,2026-05-21,MSFT,6.038086,3.152553,0.042164,2.843369,37.517770,37.517770,MA_BASELINE
5,2024-01-02,MSFT,5.896608,3.028852,0.032674,2.835083,37.408376,37.408376,MA_BASELINE
6,2026-05-21,AMZN,5.592702,2.910156,0.032487,2.650059,37.245601,37.245601,MA_BASELINE
7,2026-05-20,GOOGL,5.963348,3.178730,0.035726,2.748892,36.734463,36.734463,MA_BASELINE
8,2026-05-20,MSFT,6.040612,3.246482,0.042981,2.751148,36.300218,36.300218,MA_BASELINE
9,2024-01-03,MSFT,5.895880,3.123838,0.029364,2.742678,36.188387,36.188387,MA_BASELINE


## 6. Guardrails


In [7]:
guardrails = article_language_guardrails()
guardrails


,unsafe,safer
0,This trend predicts the next price move.,This trend summarizes the observed public seri...
1,This model is better because it has more downl...,Downloads are a public adoption proxy and shou...
2,This repo is winning because stars are rising.,"Star velocity measures developer attention, no..."
3,This pageview spike proves importance.,Pageviews measure public attention during the ...
4,This residual is a buy signal.,This residual marks an event-like deviation fr...


In [8]:
save_table(audit, "07_ai_infra_market_audit")
save_table(summary, "07_ai_infra_component_summary")
save_table(returns, "07_ai_infra_return_proxy")
save_table(events, "07_ai_infra_residual_events")
save_table(guardrails, "07_ai_infra_guardrails")


saved: examples/hot_trends/outputs/07_ai_infra_market_audit.csv
saved: examples/hot_trends/outputs/07_ai_infra_component_summary.csv
saved: examples/hot_trends/outputs/07_ai_infra_return_proxy.csv
saved: examples/hot_trends/outputs/07_ai_infra_residual_events.csv
saved: examples/hot_trends/outputs/07_ai_infra_guardrails.csv
